# Clothing segmentation with the Hugging Face Inference API

This notebook sends a single image to the pre-trained
[SegFormer B3 Clothes](https://huggingface.co/sayeed99/segformer_b3_clothes) model
through the Hugging Face Inference API, and turns its response into a segmentation mask.

The API returns one binary mask per detected class, encoded in base64.
These masks are decoded and merged into a single mask where each pixel holds
the ID of its class, the same format as the reference masks of the dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

## Configuration

Imports, and the paths used in this notebook:

* `IMAGE_PATH`: the image to segment. Change it to try another image of the dataset.
* `MASK_PATH`: the reference mask of that image, annotated by hand.
* `PREDICTED_MASK_PATH`: where the mask predicted by the model is saved, in `data/output/`.

`load_dotenv()` reads the `.env` file at the root of the project, which holds the Hugging Face token (`HF_TOKEN`).

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
from oc_p02_fashion_segmentation.hf_api import HFAPIError, get_hf_token, segment_image
from oc_p02_fashion_segmentation.masks import create_masks, save_mask
from oc_p02_fashion_segmentation.visualization import display_overlay

DATA_DIR = Path("../data/input")
IMAGE_PATH = DATA_DIR / "IMG" / "image_0.png"
MASK_PATH = DATA_DIR / "Mask" / "mask_0.png"
OUTPUT_DIR = Path("../data/output")
# Named like the dataset masks: image_0.png -> mask_0.png
PREDICTED_MASK_PATH = OUTPUT_DIR / IMAGE_PATH.name.replace("image_", "mask_")

load_dotenv()

## Query the API

The image is loaded to get its size: the masks returned by the API are resized to it in the next step.

`segment_image` sends the image to the model and returns a list with one entry per detected class: its `label` (the class name) and its `mask` (a binary PNG encoded in base64). It raises an `HFAPIError` if the token is missing or invalid, the request times out, or the API returns an error.

In [ ]:
image = Image.open(IMAGE_PATH)
image_array = np.array(image)
image_width, image_height = image.size

result = segment_image(get_hf_token(), IMAGE_PATH)

## Visualize the result

Overlay the predicted mask on the original image, with one color per class.

In [ ]:
mask_array = create_masks(result, image_width, image_height)
display_overlay(image_array, mask_array)

## Save the predicted mask

The mask is saved as PNG, a lossless format, so every pixel keeps its exact class ID. With JPEG, compression would alter pixels near class edges and turn them into other classes.

In [ ]:
save_mask(mask_array, PREDICTED_MASK_PATH)
print(f"Predicted mask saved to {PREDICTED_MASK_PATH}")